# Bayesian Test Case Reduction (TCR) for Firebase Chat QA — Pilot Notebook

## What this notebook is actually testing

The 69 Firebase Chat test cases take a long time to run manually, one by one. This notebook asks a narrower question than "which vectorizer wins": **can a model that has only seen a handful of test outcomes so far learn to guess, well enough, which of the remaining test cases are unlikely to find a bug — and stop early once it's confident nothing more is left to find?** If that works, QA would only need to run a smaller, smartly-chosen subset instead of all 69.

**Important — this is still a pilot, not a finished result.** The 13 "bugs" used here are fixed, hand-placed dummy labels on specific test-case IDs, not outcomes from a real QA run. Nothing in this notebook should be read as "this vectorizer/kernel/setting finds real Android bugs faster" — it only tells us how each modeling choice behaves on one controlled, synthetic scenario, which is the necessary first step before running this against real execution outcomes (see the reduction `README.md` for what real ground truth would take).

The engine underneath is a Gaussian Process Classifier (GPC) with a probit link — a standard way to model a yes/no outcome (bug found / not found) with calibrated uncertainty, so the model can say "I don't know yet" instead of guessing overconfidently. All nine text representations from the sibling prioritization notebook (TF-IDF, Feature Hashing, Word2Vec, GloVe, FastText, ELMo, Flair, One-Hot Encoding, Multilingual E5 Large Instruct) are compared here across eight small experiments, **EXP1 through EXP8**, each answering one specific question:

- **EXP1** — Which of the 9 text representations gives the model the most useful sense of "these two test cases are similar"?
- **EXP2** — Does the choice of similarity/kernel function (cosine vs. RBF vs. Matern vs. Rational Quadratic) matter?
- **EXP3** — Is it worth letting the model tune its own hyperparameters (MLE) instead of using fixed defaults?
- **EXP4** — Which rule for picking the next test case (Cost-UCB, EI, LogEI, UCB, PI, Thompson Sampling) finds bugs fastest?
- **EXP5** — For UCB specifically, how much should the model favor exploring uncertain cases (the beta knob)?
- **EXP6** — How sensitive is the "stop early" decision to the confidence threshold theta?
- **EXP7** — If QA can only afford to run 20, 30, 40, or 50 of the 69 cases, how does this approach compare to just picking cases at random?
- **EXP8** — Do the results above hold up if the dummy bugs are scattered randomly instead of placed in the two hand-picked spots — or was the model just getting lucky on one specific pattern?

In [ ]:
import os

# === EXPERIMENT CONFIGURATION ===
# Vectorizer selection (identical to prioritization experiment)
PAPER_VECTORISERS = (
    "TF-IDF", "Feature Hashing", "Word2Vec", "GloVe",
    "FastText", "ELMo", "Flair",
)
ADDITIONAL_VECTORISERS = ("One-Hot Encoding", "Multilingual E5 Large Instruct")
EXPERIMENT_METHODS = PAPER_VECTORISERS + ADDITIONAL_VECTORISERS
QUICK_VECTORISERS = ("TF-IDF", "Feature Hashing")

def select_vectorisers(selection: str) -> tuple[str, ...]:
    requested = [item.strip() for item in selection.split(",") if item.strip()]
    if not requested or requested == ["quick"]:
        return QUICK_VECTORISERS
    if requested == ["all"]:
        return EXPERIMENT_METHODS
    return tuple(name for name in EXPERIMENT_METHODS if name in requested)

VECTORISER_SELECTION = os.getenv("PAPER_VECTORISER_MODE", "all")
SELECTED_VECTORISERS = select_vectorisers(VECTORISER_SELECTION)
print(f"Selected vectorisers ({len(SELECTED_VECTORISERS)} total):", ", ".join(SELECTED_VECTORISERS))

# Kernel - EXP2 (env var: BO_KERNEL)
KERNEL_NAME = os.getenv("BO_KERNEL", "cosine")

# HP Tuning - EXP3 (env var: BO_HP_TUNING)
HP_TUNING = os.getenv("BO_HP_TUNING", "fixed")

# Acquisition Function - EXP4 (env var: BO_AF)
AF_NAME = os.getenv("BO_AF", "cost_ucb")

# REDUCTION PARAMETERS
STOP_THETA = float(os.getenv("BO_STOP_THETA", "0.1"))
MAX_TC = int(os.getenv("BO_MAX_TC", "35"))
SIGMA2 = float(os.getenv("BO_SIGMA2", "1.0"))
NOISE_ALPHA = float(os.getenv("BO_NOISE_ALPHA", "0.01"))
UCB_BETA = float(os.getenv("BO_UCB_BETA", "2.0"))
N_FAIRNESS_REPLICATES = int(os.getenv("BO_FAIRNESS_REPS", "10"))


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

# Persistent pretrained-weight cache (HAKUSAN / SSH JAIST): home dir is NFS-shared
# and survives across sessions; /tmp is wiped. Mirrors experiment/bayesian/prioritization.
PERSISTENT_CACHE = Path.home() / ".mas_ai_cache"
PERSISTENT_CACHE.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("GENSIM_DATA_DIR", str(PERSISTENT_CACHE / "gensim"))
os.environ.setdefault("TFHUB_CACHE_DIR", str(PERSISTENT_CACHE / "tfhub"))
os.environ.setdefault("HF_HOME", str(PERSISTENT_CACHE / "huggingface"))
os.environ.setdefault("SENTENCE_TRANSFORMERS_HOME", str(PERSISTENT_CACHE / "sentence_transformers"))
os.environ.setdefault("FLAIR_CACHE_DIR", str(PERSISTENT_CACHE / "flair"))
for _p in (PERSISTENT_CACHE / "gensim", PERSISTENT_CACHE / "tfhub", PERSISTENT_CACHE / "huggingface"):
    _p.mkdir(parents=True, exist_ok=True)

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# FIXED: map each package to its actual importable module name (was checking
# "scikit_learn", which never exists -- the import name is "sklearn" -- so this
# used to always report scikit-learn missing and always retry installing it,
# which fails hard with no internet on an HPC compute node even when it is
# already installed).
BASE_PACKAGES = {
    "openpyxl": "openpyxl",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "pandas": "pandas",
    "numpy": "numpy",
}
for module_name, pip_name in BASE_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        install(pip_name)

# FIXED: single source of truth for the vectoriser-mode toggle. The SSH JAIST
# launcher (config.env) exports HAKUSAN_VECTORISER_MODE; the vectoriser-selection
# cell above reads PAPER_VECTORISER_MODE. Previously these were two independent,
# unsynced env vars -- if only one was set, the printed mode and the packages
# actually installed would silently disagree. Now either name works, and they
# resolve to the same value.
VECTORISER_MODE = os.getenv(
    "PAPER_VECTORISER_MODE", os.getenv("HAKUSAN_VECTORISER_MODE", "all")
)
if VECTORISER_MODE == "all":
    for pkg in ["gensim", "tensorflow", "tensorflow-hub", "flair>=0.14,<0.15",
                "sentence-transformers>=2.7,<3.0", "setuptools<70"]:
        install(pkg)
    print("All vectoriser packages installed.")
else:
    print("Quick mode: only TF-IDF and Feature Hashing available.")


In [ ]:
from pathlib import Path
import hashlib, json, math, re, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.special import ndtr
from scipy.optimize import minimize
warnings.filterwarnings("ignore")

## Step 1 — Load the 69 real Firebase Chat test cases

This reads the actual QA workbook (`scenario.xlsx`) that Suitmedia uses — the same 69 cases used everywhere else in this thesis. Only the *test case text* (menu, scenario, steps, expected result) is real here; which of them "have a bug" is still the fixed dummy labeling described above, not a real QA result.

In [ ]:
import re
from pathlib import Path
import numpy as np
import pandas as pd

def parse_minutes(value):
    if pd.isna(value): return 5.0
    s = str(value).strip().lower()
    m = re.match(r"(\d+(?:\.\d+)?)\s*(?:menit|min|m)?", s)
    return float(m.group(1)) if m else 5.0

def find_workbook():
    curr = Path(".").resolve()
    search_roots = [curr] + list(curr.parents) + [
        Path(r"C:/Users/radit/Project/VisualStudioProject/Skripsi/MAS AI").resolve(),
        Path(r"D:/Kuliah/SKRIPSI/RESEARCH TEMA PROKSI/AUTOMATED TESTING ANDROID/Dokumen Suitmedia").resolve(),
        Path(r"D:/Kuliah/SKRIPSI/RESEARCH TEMA PROKSI/AUTOMATED TESTING ANDROID/Dokumen Kepake").resolve()
    ]
    for root in search_roots:
        if root.exists():
            target = root / "scenarios" / "firebase_chat" / "scenario.xlsx"
            if target.exists():
                return target
            for p in [root] + list(root.parents):
                found = list(p.glob("**/scenarios/firebase_chat/scenario.xlsx")) + list(p.glob("**/scenario.xlsx"))
                if found:
                    return found[0]
    raise FileNotFoundError("No scenario.xlsx found in standard paths.")

def load_qa_cases(path):
    df_raw = pd.read_excel(path, sheet_name=0, header=None)
    header_idx = None
    for idx, row in df_raw.iterrows():
        if any("TCS ID" in str(val) for val in row.values):
            header_idx = idx
            break
    if header_idx is not None:
        df = pd.read_excel(path, sheet_name=0, header=header_idx)
    else:
        df = df_raw
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(subset=["TCS ID"]).reset_index(drop=True)
    df = df[df["TCS ID"].astype(str).str.startswith("FC-")].reset_index(drop=True)
    df["cost_minutes"] = df.get("Time Testing", pd.Series([5.0]*len(df))).apply(parse_minutes)
    return df

WORKBOOK_PATH = find_workbook()
cases = load_qa_cases(WORKBOOK_PATH)
assert len(cases) == 69, f"Expected 69 cases, got {len(cases)}"
print(f"Loaded {len(cases)} test cases from {WORKBOOK_PATH.name}")
print(f"Total estimated time: {cases['cost_minutes'].sum():.0f} minutes")


In [ ]:
# Fixed dummy bug indices (13 dummy bugs out of 69 TC)
DUMMY_BUG_INDICES = {1, 4, 9, 13, 18, 22, 26, 30, 35, 40, 50, 57, 62}
oracle = np.array([1.0 if i in DUMMY_BUG_INDICES else 0.0 for i in range(len(cases))], dtype=float)

# Warm start: 1 TC per Menu (coverage-first)
menus = cases["Menu"].tolist()
seen_menus = set()
initial_indices = []
for i, menu in enumerate(menus):
    if menu not in seen_menus:
        seen_menus.add(menu)
        initial_indices.append(i)

print(f"Oracle: {int(oracle.sum())} dummy bugs in {len(cases)} TC")
print(f"Warm start: {len(initial_indices)} TC (1 per menu)")
print(f"Budget: max {MAX_TC} TC, stopping theta={STOP_THETA}\n")

# Transparent Output 1: Warm Start Seeds Table
print("=== WARM START INITIAL SEEDS (1 TC PER MENU) ===")
warm_df = cases.iloc[initial_indices][["TCS ID", "Menu", "Test Case Scenario", "cost_minutes"]].copy()
warm_df["Is Bug?"] = oracle[initial_indices].astype(bool)
print(warm_df.to_string(index=False))

# Transparent Output 2: Fixed Dummy Bugs Table
print("\n=== FIXED DUMMY BUGS DISTRIBUTION (13 TOTAL) ===")
bug_indices = list(DUMMY_BUG_INDICES)
bug_df = cases.iloc[bug_indices][["TCS ID", "Menu", "Test Case Scenario", "cost_minutes"]].copy()
print(bug_df.to_string(index=False))


## Step 2 — Turn each test case's text into numbers

A Gaussian Process needs to measure how "similar" two test cases are, and it can only do that once each test case's text has been converted into a vector of numbers. This step builds that matrix once per text representation (TF-IDF, Word2Vec, and so on) — nine independent matrices, one per method, so EXP1 can compare them on equal footing later.

A quick note if you're re-running this: the six representations that need pretrained embeddings (Word2Vec, GloVe, FastText, ELMo, Flair, Multilingual E5) now fail loudly and tell you exactly what's missing if the library or model isn't available — they used to quietly substitute random numbers instead, which is exactly the kind of thing that would produce a chart that *looks* fine but means nothing. See the note in that cell for how to run in a lighter "quick" mode with just TF-IDF and Feature Hashing when you don't want to install everything.

In [ ]:
TEXT_FIELDS = [
    "Menu", "Submenu 1", "Submenu 2", "Test Case Scenario", "Test Step",
    "Expected Result"
]
available_cols = [c for c in TEXT_FIELDS if c in cases.columns]
corpus = cases[available_cols].fillna("").apply(lambda r: " ".join(r.astype(str)), axis=1).tolist()

from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer
from sklearn.preprocessing import normalize

def build_matrices(methods, corpus):
    matrices = {}
    for method in methods:
        if method == "TF-IDF":
            vec = TfidfVectorizer(max_features=500)
            X = vec.fit_transform(corpus).toarray()
        elif method == "Feature Hashing":
            vec = HashingVectorizer(n_features=256, alternate_sign=False)
            X = vec.transform(corpus).toarray()
        elif method == "One-Hot Encoding":
            tokens = [set(doc.lower().split()) for doc in corpus]
            vocab = sorted(set(w for t in tokens for w in t))
            X = np.array([[1.0 if w in t else 0.0 for w in vocab] for t in tokens])
        elif method == "Word2Vec":
            try:
                import gensim.downloader as api
                w2v = api.load("word2vec-google-news-300")
                X = np.array([np.mean([w2v[w] for w in doc.lower().split() if w in w2v] or [np.zeros(300)], axis=0) for doc in corpus])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build Word2Vec embeddings: {exc}. Install gensim "
                    "(pip install gensim), or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        elif method == "GloVe":
            try:
                import gensim.downloader as api
                glove = api.load("glove-wiki-gigaword-100")
                X = np.array([np.mean([glove[w] for w in doc.lower().split() if w in glove] or [np.zeros(100)], axis=0) for doc in corpus])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build GloVe embeddings: {exc}. Install gensim "
                    "(pip install gensim), or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        elif method == "FastText":
            try:
                import gensim.downloader as api
                ft = api.load("fasttext-wiki-news-subwords-300")
                X = np.array([np.mean([ft[w] for w in doc.lower().split() if w in ft] or [np.zeros(300)], axis=0) for doc in corpus])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build FastText embeddings: {exc}. Install gensim "
                    "(pip install gensim), or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        elif method == "ELMo":
            try:
                # google/elmo/3 on TF-Hub is published in the legacy TF1 Hub-module
                # format, not a native TF2 SavedModel. hub.load() (the TF2 API) can
                # open the file, but the object it returns has no usable __call__ or
                # .signatures for this module -- that is exactly what the
                # "'AutoTrackable' object is not callable" error means. The correct
                # API for a TF1-format module is hub.Module() run inside a TF1
                # compatibility session (this is TF-Hub's own documented fix for
                # this exact error class -- see tensorflow.org/hub/common_issues).
                import tensorflow.compat.v1 as tf1
                import tensorflow_hub as hub
                tf1.disable_eager_execution()
                elmo_module = hub.Module("https://tfhub.dev/google/elmo/3", trainable=False)
                embed_op = elmo_module(corpus, signature="default", as_dict=True)["default"]
                with tf1.Session() as sess:
                    sess.run(tf1.global_variables_initializer())
                    sess.run(tf1.tables_initializer())
                    X = sess.run(embed_op)
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build ELMo embeddings: {exc}. This module needs the "
                    "TF1-compat hub.Module API (tensorflow_hub still ships it, but it "
                    "requires eager execution to be disabled for the rest of this "
                    "kernel), or run with PAPER_VECTORISER_MODE=quick to skip this "
                    "vectoriser instead of silently substituting random noise."
                ) from exc
        elif method == "Flair":
            try:
                from flair.embeddings import WordEmbeddings, DocumentPoolEmbeddings
                from flair.data import Sentence
                document_embeddings = DocumentPoolEmbeddings([WordEmbeddings('glove')])
                sentences = [Sentence(doc) for doc in corpus]
                document_embeddings.embed(sentences)
                X = np.array([s.embedding.cpu().numpy() for s in sentences])
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build Flair embeddings: {exc}. Install flair, or run "
                    "with PAPER_VECTORISER_MODE=quick to skip this vectoriser instead "
                    "of silently substituting random noise."
                ) from exc
        elif method == "Multilingual E5 Large Instruct":
            try:
                from sentence_transformers import SentenceTransformer
                model = SentenceTransformer('intfloat/multilingual-e5-large')
                X = model.encode(corpus, show_progress_bar=False)
            except Exception as exc:
                raise RuntimeError(
                    f"Failed to build Multilingual E5 embeddings: {exc}. Install "
                    "sentence-transformers, or run with PAPER_VECTORISER_MODE=quick "
                    "to skip this vectoriser instead of silently substituting random noise."
                ) from exc
        else:
            X = np.random.RandomState(48).randn(len(corpus), 64)
        X = normalize(X, norm="l2")
        matrices[method] = X
        print(f"  {method:32s}: shape {X.shape}")
    return matrices

print(f"Building matrices for all selected methods: {SELECTED_VECTORISERS}")
matrices = build_matrices(SELECTED_VECTORISERS, corpus)


## Step 3 — The actual model: GPC with a probit link, plus the stopping rule

This is the core of the notebook, so it's worth spelling out in plain terms.

**Why not just average a plain regression model?** A bug outcome is binary (found / not found), not a continuous number, so a standard Gaussian Process (built for continuous, noisy measurements) is the wrong tool. Gaussian Process **Classification** fixes that: it assumes each test case has some hidden "risk score" (called `f`), and the *probit* function (the standard normal cumulative distribution, `Φ`) squashes that risk score into a proper probability between 0 and 1. Because there's no simple formula for exactly the right hidden score, the model uses the **Laplace approximation** — a small loop of Newton's-method updates (matching the textbook derivation in Rasmussen & Williams, *Gaussian Processes for Machine Learning*, 2006, Algorithm 3.1) that finds the most likely hidden score given whatever outcomes have been observed so far.

One detail worth calling out because it was a real bug caught during review: converting that hidden score back into a probability the model reports has an **exact** formula for a probit model — `Φ(mean / sqrt(1 + variance))` — no approximation needed. An earlier version of this notebook used a different constant (`π/8`) borrowed from a formula for *logistic* models, which doesn't apply here and made the model's confidence systematically wrong, especially when it was still uncertain. That's fixed now; the numbers here use the exact formula.

**How the next test case gets picked.** Once the model has a predicted probability and an uncertainty for every remaining test case, it scores each one with an acquisition function — by default, "predicted risk + a bonus for uncertainty, adjusted for how expensive that test case is to run" (Cost-UCB). EXP4 tries five alternatives to this rule.

**When it decides to stop early.** After each pick, the model checks: for every test case still left, is even its most optimistic guess (mean + a safety margin) below a small threshold theta? If yes for all of them, it stops — the whole point of "reduction" rather than just "reordering." EXP6 checks how sensitive that stopping point is to the choice of theta.

In [ ]:
import numpy as np
from scipy.special import ndtr
from scipy.optimize import minimize as sp_minimize

# ── Kernel functions ─────────────────────────────────────────────────────────────

def _cosine_kernel(X, Y):
    return X @ Y.T

def _rbf_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    return np.exp(-0.5 * np.sum(diff**2, axis=-1) / length_scale**2)

def _matern32_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    v = r / length_scale
    return (1 + np.sqrt(3)*v) * np.exp(-np.sqrt(3)*v)

def _matern52_kernel(X, Y, length_scale=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r = np.sqrt(np.sum(diff**2, axis=-1))
    v = r / length_scale
    return (1 + np.sqrt(5)*v + 5*v**2/3) * np.exp(-np.sqrt(5)*v)

def _rq_kernel(X, Y, length_scale=1.0, alpha=1.0):
    diff = X[:, None, :] - Y[None, :, :]
    r2 = np.sum(diff**2, axis=-1)
    return (1 + r2 / (2*alpha*length_scale**2))**(-alpha)

def get_kernel(name, X, Y, sigma2=1.0, length_scale=1.0):
    if name == "cosine":
        return sigma2 * _cosine_kernel(X, Y)
    elif name == "rbf":
        return sigma2 * _rbf_kernel(X, Y, length_scale)
    elif name == "matern32":
        return sigma2 * _matern32_kernel(X, Y, length_scale)
    elif name == "matern52":
        return sigma2 * _matern52_kernel(X, Y, length_scale)
    elif name == "rq":
        return sigma2 * _rq_kernel(X, Y, length_scale)
    else:
        raise ValueError(f"Unknown kernel: {name}")

# ── Probit derivatives (Laplace GPC) ────────────────────────────────────────────

def _probit_derivatives(f, y):
    yf = y * f
    phi = ndtr(yf)
    phi = np.clip(phi, 1e-10, 1 - 1e-10)
    pdf = np.exp(-0.5 * yf**2) / np.sqrt(2 * np.pi)
    grad = y * pdf / phi
    W = (pdf / phi)**2 + yf * pdf / phi
    return grad, W

# ── MLE for kernel hyperparameters (EXP3) ───────────────────────────────────────

def _neg_lml(log_params, X, y, kernel_name, noise):
    sigma2 = np.exp(log_params[0])
    length_scale = np.exp(log_params[1]) if len(log_params) > 1 else 1.0
    n = len(y)
    K = get_kernel(kernel_name, X, X, sigma2, length_scale)
    K += noise * np.eye(n)
    try:
        L = np.linalg.cholesky(K)
        alpha = np.linalg.solve(L.T, np.linalg.solve(L, y))
        lml = -0.5 * y @ alpha - np.sum(np.log(np.diag(L))) - 0.5 * n * np.log(2*np.pi)
        return -lml
    except np.linalg.LinAlgError:
        return 1e10

def tune_hyperparams(X, y, kernel_name, noise, n_restarts=5):
    best_val, best_params = np.inf, [0.0, 0.0]
    for _ in range(n_restarts):
        x0 = np.random.randn(2)
        res = sp_minimize(_neg_lml, x0, args=(X, y, kernel_name, noise),
                          method="L-BFGS-B")
        if res.fun < best_val:
            best_val = res.fun
            best_params = res.x
    return np.exp(best_params[0]), np.exp(best_params[1])

# ── GPC predict (Laplace approximation) ──────────────────────────────────────────

def _gp_predict(train_x, train_y, cand_x,
                sigma2=SIGMA2, noise=NOISE_ALPHA,
                kernel_name=KERNEL_NAME, hp_tuning=HP_TUNING):
    if hp_tuning == "mle" and len(train_y) >= 5:
        sigma2, length_scale = tune_hyperparams(train_x, train_y*2-1, kernel_name, noise)
    else:
        length_scale = 1.0

    train_y_pm = train_y * 2 - 1  # {0,1} -> {-1,+1}
    n = len(train_y)
    K = get_kernel(kernel_name, train_x, train_x, sigma2, length_scale)
    K += noise * np.eye(n)

    # Laplace: Newton iterations
    f = np.zeros(n)
    for _ in range(20):
        grad, W = _probit_derivatives(f, train_y_pm)
        W_safe = np.maximum(W, 1e-8)
        W_sqrt = np.sqrt(W_safe)
        B = np.eye(n) + W_sqrt[:, None] * K * W_sqrt[None, :]
        L = np.linalg.cholesky(B + 1e-8 * np.eye(n))
        b = W_safe * f + grad
        v = np.linalg.solve(L, W_sqrt * (K @ b))
        f_new = K @ (b - W_sqrt * np.linalg.solve(L.T, v))
        if np.max(np.abs(f_new - f)) < 1e-6:
            f = f_new
            break
        f = f_new

    grad, W = _probit_derivatives(f, train_y_pm)
    W_safe = np.maximum(W, 1e-8)
    W_sqrt = np.sqrt(W_safe)
    B = np.eye(n) + W_sqrt[:, None] * K * W_sqrt[None, :]
    L = np.linalg.cholesky(B + 1e-8 * np.eye(n))

    K_star = get_kernel(kernel_name, cand_x, train_x, sigma2, length_scale)
    mu = K_star @ grad
    v = np.linalg.solve(L, W_sqrt[:, None] * K_star.T)
    K_ss = np.diag(get_kernel(kernel_name, cand_x, cand_x, sigma2, length_scale))
    var = np.maximum(K_ss - np.sum(v**2, axis=0), 1e-8)
    std = np.sqrt(var)

    # Squash to [0,1] probability
    kappa = 1.0 / np.sqrt(1 + var)  # FIXED: exact probit squashing (GPML eq 3.25);
    # the previous np.pi/8 constant is the MacKay logistic-approximation-to-probit
    # formula, wrong for a model whose own likelihood already IS probit.
    mean_prob = ndtr(kappa * mu)
    phi_star = np.exp(-0.5 * (kappa * mu)**2) / np.sqrt(2 * np.pi)
    std_prob = np.maximum(phi_star * kappa * std, 1e-6)
    return mean_prob, std_prob

# ── Acquisition functions ─────────────────────────────────────────────────────────

def _cost_ucb(mean, std, cost, beta=UCB_BETA):
    return (mean + beta * std) / np.sqrt(np.maximum(cost, 1.0))

def _ei(mean, std, best, xi=0.01):
    z = (mean - best - xi) / np.maximum(std, 1e-8)
    return (mean - best - xi) * ndtr(z) + std * np.exp(-0.5*z**2)/np.sqrt(2*np.pi)

def _logei(mean, std, best, xi=0.01):
    ei = _ei(mean, std, best, xi)
    return np.log(np.maximum(ei, 1e-30))

def _ucb(mean, std, beta=UCB_BETA):
    return mean + beta * std

def _pi(mean, std, best, xi=0.01):
    return ndtr((mean - best - xi) / np.maximum(std, 1e-8))

def _thompson(mean, std, rng):
    return rng.normal(mean, std)

def get_af_score(af_name, mean, std, cost, best, rng):
    if af_name == "cost_ucb":
        return _cost_ucb(mean, std, cost)
    elif af_name == "ei":
        return _ei(mean, std, best)
    elif af_name == "logei":
        return _logei(mean, std, best)
    elif af_name == "ucb":
        return _ucb(mean, std)
    elif af_name == "pi":
        return _pi(mean, std, best)
    elif af_name == "ts":
        return _thompson(mean, std, rng)
    else:
        raise ValueError(f"Unknown AF: {af_name}")

# ── Stopping criterion ─────────────────────────────────────────────────────────────

def _should_stop(mean, std, beta, theta):
    """Return True jika semua kandidat tersisa punya upper bound < theta."""
    upper_bounds = mean + beta * std
    return bool(np.all(upper_bounds < theta))

# ── Reduction loop (CORE PERBEDAAN dari prioritization) ────────────────────────────

def run_reduction_loop(matrix, ids, menus, costs, oracle,
                       initial_indices,
                       max_tc=MAX_TC,
                       stop_theta=STOP_THETA,
                       kernel_name=KERNEL_NAME,
                       af_name=AF_NAME,
                       hp_tuning=HP_TUNING,
                       sigma2=SIGMA2,
                       noise=NOISE_ALPHA,
                       beta=UCB_BETA,
                       rng=None):
    if rng is None:
        rng = np.random.default_rng(42)

    n = len(ids)
    selected = list(initial_indices)
    observed_oracle = {i: oracle[i] for i in selected}
    total_bugs = int(oracle.sum())

    history = []  # list of dicts per iteration

    for step in range(max_tc - len(initial_indices)):
        candidates = [i for i in range(n) if i not in set(selected)]
        if not candidates:
            break

        train_x = matrix[selected]
        train_y = np.array([observed_oracle[i] for i in selected])
        cand_x = matrix[candidates]
        cand_costs = np.array([costs[i] for i in candidates])

        mean, std = _gp_predict(train_x, train_y, cand_x,
                                sigma2=sigma2, noise=noise,
                                kernel_name=kernel_name, hp_tuning=hp_tuning)

        # Stopping check
        if _should_stop(mean, std, beta, stop_theta):
            break

        # AF selection
        best_so_far = float(np.mean(train_y)) if train_y.sum() > 0 else 0.0
        scores = get_af_score(af_name, mean, std, cand_costs, best_so_far, rng)
        chosen_local = int(np.argmax(scores))
        chosen = candidates[chosen_local]

        selected.append(chosen)
        observed_oracle[chosen] = oracle[chosen]

        bugs_found = sum(observed_oracle[i] for i in selected)
        recall = bugs_found / total_bugs if total_bugs > 0 else 0.0
        total_cost = sum(costs[i] for i in selected)

        history.append({
            "step": len(selected),
            "chosen_id": ids[chosen],
            "bugs_found": bugs_found,
            "bug_recall": recall,
            "total_cost_min": total_cost,
            "stopped_early": False
        })

    # Mark if stopped by theta (not by max_tc)
    stopped_by_theta = len(selected) < max_tc and len(selected) < n
    bugs_found = sum(observed_oracle[i] for i in selected)
    recall = bugs_found / total_bugs if total_bugs > 0 else 0.0

    return {
        "selected_indices": selected,
        "n_selected": len(selected),
        "bugs_found": bugs_found,
        "bug_recall": recall,
        "stopped_by_theta": stopped_by_theta,
        "total_cost_min": sum(costs[i] for i in selected),
        "history": history
    }

print("GPC + Reduction loop defined.")
print(f"  kernel={KERNEL_NAME}, af={AF_NAME}, hp={HP_TUNING}")
print(f"  stop_theta={STOP_THETA}, max_tc={MAX_TC}")


## EXP1 — Which text representation works best? (RQ1)

**Question:** out of the 9 ways to turn a test case's text into numbers, which one gives the model the clearest signal for finding the dummy bugs with the fewest test cases run?

**What happens in this cell:** the same reduction loop runs once per representation, with every other setting held fixed (cosine kernel, Cost-UCB, stop at theta = 0.1, budget capped at 35 of the 69 cases). What's compared afterward is simply: how many test cases did it end up running, and what fraction of the 13 dummy bugs did it catch.

In [ ]:
costs = cases["cost_minutes"].values
ids = cases["TCS ID"].tolist()

reduction_results = {}
print("=== EXP1: EXECUTING REDUCTION LOOP FOR ALL 9 VECTORISERS ===\n")
for method_name, matrix in matrices.items():
    result = run_reduction_loop(
        matrix=matrix,
        ids=ids,
        menus=menus,
        costs=costs,
        oracle=oracle,
        initial_indices=initial_indices,
        max_tc=MAX_TC,
        stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME,
        af_name=AF_NAME,
        hp_tuning=HP_TUNING,
        rng=np.random.default_rng(42)
    )
    reduction_results[method_name] = result
    print(f"[{method_name:32s}] Executed {int(result['n_selected']):2d} TC | "
          f"Bugs Found: {int(result['bugs_found']):2d}/13 ({result['bug_recall']:.1%}) | "
          f"Cost: {result['total_cost_min']:.0f} min | "
          f"Stopped Early: {result['stopped_by_theta']}")
    
    # Print first 5 step-by-step choices for transparency
    print(f"  Step-by-step selected TC sequence (first 5): {[h['chosen_id'] for h in result['history'][:5]]}...")


## EXP1 results, in a table and a chart

This saves the EXP1 numbers to `results/` as CSV/JSON, and draws a bar chart: bars show how many test cases each representation ran before stopping, the dashed line shows how many of the 13 dummy bugs it caught.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import json, zipfile

os.makedirs("results", exist_ok=True)

summary_rows = []
for method, r in reduction_results.items():
    summary_rows.append({
        "Method": method,
        "TC Dijalankan": r["n_selected"],
        "TC Dijalankan (%)": f"{r['n_selected']/69*100:.1f}%",
        "Bug Recall": f"{r['bug_recall']:.1%}",
        "Bugs Found": r["bugs_found"],
        "Cost (menit)": f"{r['total_cost_min']:.0f}",
        "Stopped by theta": r["stopped_by_theta"]
    })

df_summary = pd.DataFrame(summary_rows)
print(df_summary.to_string(index=False))
df_summary.to_csv("results/exp1_vectorizer_summary.csv", index=False)
with open('results/exp1_reduction_results.json', 'w') as f:
    json.dump({k: {rk: (rv.tolist() if isinstance(rv, np.ndarray) else bool(rv) if isinstance(rv, (bool, np.bool_)) else int(rv) if isinstance(rv, (int, np.integer)) else float(rv) if isinstance(rv, (float, np.floating)) else rv) for rk, rv in v.items()} for k, v in reduction_results.items()}, f, indent=2)

fig, ax = plt.subplots(figsize=(8, 5))
methods = list(reduction_results.keys())
recalls = [reduction_results[m]["bug_recall"] for m in methods]
n_selected = [reduction_results[m]["n_selected"] for m in methods]
colors = sns.color_palette("tab10", len(methods))
bars = ax.bar(methods, n_selected, color=colors)
ax2 = ax.twinx()
ax2.plot(methods, recalls, "D--k", label="Bug Recall")
ax2.set_ylim(0, 1.1)
ax2.set_ylabel("Bug Recall")
ax.set_ylabel("TC Dijalankan")
ax.set_title(f"Reduction: TC Dijalankan vs Bug Recall\n(theta={STOP_THETA}, max_tc={MAX_TC})")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("results/reduction_summary.png", dpi=100)
plt.show()
print("EXP1 CSV & JSON saved to results/")


## EXP2 — Does the similarity function (kernel) matter? (RQ2)

**Question:** cosine similarity is the default, but is it actually the best way to measure "how alike are these two test cases"? This tries four alternatives — RBF, Matern 3/2, Matern 5/2, and Rational Quadratic — all standard covariance functions from the Gaussian Process literature, each with a different assumption about how quickly similarity should fall off with distance.

**What happens in this cell:** same reduction loop, one representation held fixed (whichever came first in EXP1), kernel swapped each run.

In [ ]:
KERNEL_OPTIONS = ['cosine', 'rbf', 'matern32', 'matern52', 'rq']
best_method = list(matrices.keys())[0]
best_matrix = matrices[best_method]
print(f"EXP2: Kernel sweep using vectorizer '{best_method}'")

kernel_results = []
for k_name in KERNEL_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=menus, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=MAX_TC, stop_theta=STOP_THETA,
        kernel_name=k_name, af_name=AF_NAME, hp_tuning=HP_TUNING, rng=np.random.default_rng(42)
    )
    kernel_results.append({
        "kernel": k_name,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"],
        "stopped_by_theta": r["stopped_by_theta"]
    })
    print(f"  kernel={k_name:10s}: {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_kernel = pd.DataFrame(kernel_results)
df_kernel.to_csv("results/exp2_kernel_sweep.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(df_kernel["kernel"], df_kernel["bug_recall"], color="teal")
ax.set_ylabel("Bug Recall")
ax.set_ylim(0, 1.1)
ax.set_title(f"EXP2: Kernel Function Comparison (vectorizer={best_method})")
plt.tight_layout()
plt.savefig("results/exp2_kernel_sweep.png", dpi=100)
plt.show()
print(df_kernel.to_string(index=False))


## EXP3 — Is it worth tuning the kernel's hyperparameters? (RQ3)

**Question:** instead of using a fixed length-scale of 1.0, would letting the model estimate its own hyperparameters from the data (via Marginal Likelihood Estimation, optimized with L-BFGS-B) actually help, or is it not worth the extra computation?

**What happens in this cell:** the same loop runs twice — once with fixed hyperparameters, once re-tuning them at every step — and compares the outcomes.

In [ ]:
best_method = list(matrices.keys())[0]
best_matrix = matrices[best_method]
HP_OPTIONS = ['fixed', 'mle']
print(f"EXP3: HP Tuning sweep using vectorizer '{best_method}'")

hp_results = []
for hp_mode in HP_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=menus, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=MAX_TC, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name=AF_NAME, hp_tuning=hp_mode, rng=np.random.default_rng(42)
    )
    hp_results.append({
        "hp_tuning": hp_mode,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"],
        "stopped_by_theta": r["stopped_by_theta"]
    })
    print(f"  hp_tuning={hp_mode:8s}: {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_hp = pd.DataFrame(hp_results)
df_hp.to_csv("results/exp3_hp_tuning_sweep.csv", index=False)
print(df_hp.to_string(index=False))


## EXP4 — Which rule for picking the next test case works best? (RQ4)

**Question:** Cost-UCB is the default acquisition rule, but there are five other standard choices from the Bayesian optimization literature: Expected Improvement (EI), a numerically safer version of it (LogEI), plain UCB (no cost adjustment), Probability of Improvement (PI), and Thompson Sampling. Which one actually finds bugs fastest for this kind of problem?

**What happens in this cell:** same setup, acquisition rule swapped each run.

In [ ]:
best_method = list(matrices.keys())[0]
best_matrix = matrices[best_method]
AF_OPTIONS = ['cost_ucb', 'ei', 'logei', 'ucb', 'pi', 'ts']
print(f"EXP4: Acquisition function sweep using vectorizer '{best_method}'")

af_results = []
for af_opt in AF_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=menus, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=MAX_TC, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name=af_opt, hp_tuning=HP_TUNING, rng=np.random.default_rng(42)
    )
    af_results.append({
        "acquisition_fn": af_opt,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"],
        "stopped_by_theta": r["stopped_by_theta"]
    })
    print(f"  af={af_opt:10s}: {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_af = pd.DataFrame(af_results)
df_af.to_csv("results/exp4_af_sweep.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(df_af["acquisition_fn"], df_af["bug_recall"], color="coral")
ax.set_ylabel("Bug Recall")
ax.set_ylim(0, 1.1)
ax.set_title(f"EXP4: Acquisition Function Comparison (vectorizer={best_method})")
plt.tight_layout()
plt.savefig("results/exp4_af_sweep.png", dpi=100)
plt.show()
print(df_af.to_string(index=False))


## EXP5 — How much should UCB favor exploring the unknown? (RQ5)

**Question:** UCB's behavior is controlled by one knob, beta — a higher value makes it chase uncertain/unfamiliar test cases more; a lower value makes it stick closer to what looks risky based on what's already been seen. This sweeps beta across {1.0, 2.0, 3.0, 4.0} to see where that trade-off lands best.

In [ ]:
best_method = list(matrices.keys())[0]
best_matrix = matrices[best_method]
BETA_OPTIONS = [1.0, 2.0, 3.0, 4.0]
print(f"EXP5: Beta sweep using vectorizer '{best_method}'")

beta_results = []
for b_val in BETA_OPTIONS:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=menus, costs=costs, oracle=oracle,
        initial_indices=initial_indices, max_tc=MAX_TC, stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME, af_name="ucb", hp_tuning=HP_TUNING, beta=b_val, rng=np.random.default_rng(42)
    )
    beta_results.append({
        "beta": b_val,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"],
        "stopped_by_theta": r["stopped_by_theta"]
    })
    print(f"  beta={b_val:.1f}: {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_beta = pd.DataFrame(beta_results)
df_beta.to_csv("results/exp5_beta_sweep.csv", index=False)
print(df_beta.to_string(index=False))


## EXP6 — How sensitive is the stopping point to theta? (RQ6)

**Question:** the model stops early once it's confident (upper bound below theta) that nothing's left to find. A stricter theta (e.g. 0.05) should mean running more tests but rarely stopping too soon; a looser one (0.3) should mean stopping sooner but risking missed bugs. This sweeps theta across {0.05, 0.1, 0.2, 0.3} to show that trade-off directly, in the chart below.

In [ ]:
THETA_VALUES = [0.05, 0.1, 0.2, 0.3]

best_method = list(matrices.keys())[0]
best_matrix = matrices[best_method]
print(f"EXP6: theta sweep menggunakan vectorizer '{best_method}'")

theta_results = []
for theta in THETA_VALUES:
    r = run_reduction_loop(
        matrix=best_matrix,
        ids=ids,
        menus=menus,
        costs=costs,
        oracle=oracle,
        initial_indices=initial_indices,
        max_tc=MAX_TC,
        stop_theta=theta,
        kernel_name=KERNEL_NAME,
        af_name=AF_NAME,
        rng=np.random.default_rng(42)
    )
    theta_results.append({
        "theta": theta,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"],
        "stopped_by_theta": r["stopped_by_theta"]
    })
    print(f"  theta={theta}: {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_theta = pd.DataFrame(theta_results)
df_theta.to_csv("results/exp6_theta_sweep.csv", index=False)

fig, ax1 = plt.subplots(figsize=(7, 4))
ax1.plot(df_theta["theta"], df_theta["TC_run"], "o-b", label="TC Dijalankan")
ax1.set_xlabel("Stopping Threshold (theta)")
ax1.set_ylabel("TC Dijalankan", color="b")
ax2 = ax1.twinx()
ax2.plot(df_theta["theta"], df_theta["bug_recall"], "s--r", label="Bug Recall")
ax2.set_ylabel("Bug Recall", color="r")
ax2.set_ylim(0, 1.1)
ax1.set_title(f"EXP6: Theta Sweep (vectorizer={best_method})")
plt.tight_layout()
plt.savefig("results/exp6_theta_sweep.png", dpi=100)
plt.show()
print(df_theta.to_string(index=False))


## EXP7 — How much better is this than just picking test cases at random? (RQ7)

**Question:** if QA genuinely can't run all 69 and has to pick a fixed number in advance — 20, 30, 40, or 50 — does the Bayesian approach actually beat just grabbing that many test cases at random? This is the most important sanity check in the notebook: a smart-looking method that doesn't clearly beat random selection isn't worth the added complexity.

**What happens in this cell:** the reduction loop runs once per budget size, and is compared against the *average* of 20 independent random selections at each of those same budget sizes — one random draw could get lucky or unlucky, so averaging over 20 gives a fair baseline.

In [ ]:
best_method = list(matrices.keys())[0]
best_matrix = matrices[best_method]
BUDGET_VALUES = [20, 30, 40, 50]

print(f"EXP7: Budget sweep using vectorizer '{best_method}'")

budget_results = []
for max_tc in BUDGET_VALUES:
    r = run_reduction_loop(
        matrix=best_matrix, ids=ids, menus=menus, costs=costs, oracle=oracle,
        initial_indices=initial_indices,
        max_tc=max_tc,
        stop_theta=STOP_THETA,
        kernel_name=KERNEL_NAME,
        af_name=AF_NAME,
        rng=np.random.default_rng(42)
    )
    budget_results.append({
        "max_tc": max_tc,
        "TC_run": r["n_selected"],
        "bug_recall": r["bug_recall"],
        "bugs_found": r["bugs_found"],
        "cost_min": r["total_cost_min"]
    })
    print(f"  max_tc={max_tc}: ran {r['n_selected']} TC, recall={r['bug_recall']:.1%}")

df_budget = pd.DataFrame(budget_results)
df_budget.to_csv("results/exp7_budget_sweep.csv", index=False)

random_recalls = []
rng_base = np.random.default_rng(0)
for max_tc in BUDGET_VALUES:
    recalls_seed = []
    for seed in range(20):
        rng_s = np.random.default_rng(seed)
        idx = rng_s.choice(69, size=max_tc, replace=False)
        recalls_seed.append(oracle[idx].sum() / oracle.sum())
    random_recalls.append(np.mean(recalls_seed))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df_budget["max_tc"], df_budget["bug_recall"], "o-b", label="BO Reduction")
ax.plot(BUDGET_VALUES, random_recalls, "s--r", label="Random Baseline (20-seed avg)")
ax.axhline(1.0, color="gray", linestyle=":", label="Full suite (69 TC)")
ax.set_xlabel("Budget (max TC)")
ax.set_ylabel("Bug Recall")
ax.set_ylim(0, 1.1)
ax.set_title(f"EXP7: Budget Sweep (theta={STOP_THETA})")
ax.legend()
plt.tight_layout()
plt.savefig("results/exp7_budget_sweep.png", dpi=100)
plt.show()
print(df_budget.to_string(index=False))


## EXP8 — Was EXP1-EXP7 just luck from where the dummy bugs happened to sit? (RQ8)

**Question:** the 13 dummy bugs used everywhere above were placed by hand, in two specific thematic clusters. Before trusting any of the results above, it's worth checking: do the same conclusions hold if the "bugs" are scattered completely randomly instead? If a representation only looks good because it happens to match the hand-picked pattern, that wouldn't say anything about how it'd perform on a real, unknown bug distribution.

**What happens in this cell:** 10 independent replicates, each with a fresh random placement of 13 dummy bugs, run through every representation. No single "winner" is declared here on purpose — this section exists to stress-test the earlier results, not to produce a new leaderboard.

In [ ]:
def draw_uniform_oracle(case_count, bug_count, random_state):
    rng_f = np.random.default_rng(random_state)
    oracle_f = np.zeros(case_count)
    idx = rng_f.choice(case_count, size=bug_count, replace=False)
    oracle_f[idx] = 1.0
    return oracle_f

print("=== EXP8: EXECUTING FAIRNESS CHECK ACROSS 10 SYNTHETIC BUG ORACLES ===\n")
fairness_rows = []
for method_name, matrix in matrices.items():
    for rep in range(N_FAIRNESS_REPLICATES):
        oracle_f = draw_uniform_oracle(69, 13, rep)
        r = run_reduction_loop(
            matrix=matrix,
            ids=ids,
            menus=menus,
            costs=costs,
            oracle=oracle_f,
            initial_indices=initial_indices,
            max_tc=MAX_TC,
            stop_theta=STOP_THETA,
            kernel_name=KERNEL_NAME,
            af_name=AF_NAME,
            rng=np.random.default_rng(rep)
        )
        fairness_rows.append({
            "method": method_name,
            "rep": rep,
            "n_selected": r["n_selected"],
            "bug_recall": r["bug_recall"],
            "bugs_found": r["bugs_found"]
        })

df_fairness = pd.DataFrame(fairness_rows)
df_fair_summary = df_fairness.groupby("method").agg(
    mean_recall=("bug_recall", "mean"),
    std_recall=("bug_recall", "std"),
    mean_bugs=("bugs_found", "mean"),
    mean_tc=("n_selected", "mean")
).round(3)

print("=== EXP8 FAIRNESS SUMMARY TABLE (10 REPLICATES PER METHOD) ===")
print(df_fair_summary.to_string())

os.makedirs("results", exist_ok=True)
df_fairness.to_csv("results/fairness_reduction.csv", index=False)
df_fair_summary.to_csv("results/fairness_reduction_summary.csv")
print("\nSaved fairness_reduction.csv and fairness_reduction_summary.csv to results/")


## How to read all of this

Each EXP section above answers one narrow question in isolation, with everything else held fixed at its default. None of them, alone or together, prove which real-world setting is "best" — they show how the model *behaves* under each choice, on one synthetic, controlled scenario. Treat this notebook as a methodology check that has to pass before spending real QA time and API budget on the same pipeline against real execution outcomes (see the reduction folder's `README.md` for what that next step needs).

## Packaging everything up

Bundles every CSV, JSON, and PNG produced above into one zip file, so the full EXP1-EXP8 run can be handed off or archived in one piece.

In [ ]:
import zipfile
from pathlib import Path

results_dir = Path("results")
zip_path = Path("reduction_experiment_results.zip")

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for file in results_dir.glob('*'):
        if file.is_file():
            zipf.write(file, arcname=file.name)
            print(f"Zipped: {file.name}")

print(f"\nAll EXP1-EXP8 logs, CSVs, JSONs, and PNGs packaged into: {zip_path.resolve()}")
